# Module 4: Portfolio Optimization

**QuantVerse** ? Quantitative Portfolio Intelligence System

---

## Objectives

Construct optimized portfolios using production methodologies and compare their characteristics:

1. **Equal Weight (1/N)** ? benchmark
2. **Markowitz Mean-Variance** ? minimum variance, maximum Sharpe, efficient frontier
3. **Hierarchical Risk Parity (HRP)** ? correlation hierarchy without covariance matrix inversion
4. **Risk Parity (ERC)** ? equal risk contribution
5. **Inverse Volatility** ? simple volatility-scaled allocation
6. **Mean-CVaR** ? tail-risk aware optimization

Black-Litterman is excluded from the production workflow unless dated, sourced investor views are supplied.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, logging, sys, os, json, pickle

sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')
print('Setup complete.')

In [ ]:
# Load data
data_dir = '../data/processed'
daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

# Investable only
signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]
returns = daily_returns[investable].dropna()

# Load covariance from Module 3 (or compute fresh)
# NOTE: Module 3 saves DAILY covariance — must annualize (*252) for optimization
cov_path = f'{data_dir}/covariance_lw.parquet'
if os.path.exists(cov_path):
    cov_annual = pd.read_parquet(cov_path) * 252  # Annualize daily covariance
    print('Loaded Ledoit-Wolf covariance from Module 3 (annualized: daily * 252)')
else:
    from sklearn.covariance import LedoitWolf
    lw = LedoitWolf().fit(returns.values)
    cov_annual = pd.DataFrame(lw.covariance_ * 252, index=investable, columns=investable)
    print('Computed Ledoit-Wolf covariance from returns')

# Expected returns (annualized historical mean)
# NOTE: Historical mean is a noisy estimator of forward returns (Merton 1980).
# We use it here as a baseline; walk-forward validation
# backtesting (Module 7) address this limitation explicitly.
expected_returns = returns.mean() * 252

print(f'Assets: {len(investable)}, Obs: {len(returns)}')
print(f'Date range: {returns.index[0].date()} to {returns.index[-1].date()}')

## 1. Equal Weight Benchmark (1/N)

In [ ]:
import json
from pathlib import Path

from project.config import load_config
from project.optimization import MeanVarianceOptimizer, PortfolioConstraints

metadata_path = Path('data/processed/run_metadata.json')
risk_free_rate = json.loads(metadata_path.read_text(encoding='utf-8'))['risk_free_rate'] if metadata_path.exists() else load_config('configs/base.yaml').pipeline_kwargs()['fallback_risk_free_rate']
mv = MeanVarianceOptimizer(expected_returns, cov_annual, risk_free_rate=risk_free_rate)
ew = mv.equal_weight()

print(f"Equal Weight: Return={ew['return']:.2%}, Vol={ew['volatility']:.2%}, Sharpe={ew['sharpe']:.2f}")

## 2. Markowitz Mean-Variance Optimization

In [ ]:
constraints = PortfolioConstraints.default_long_only(max_weight=0.20)

min_var = mv.minimum_variance(constraints)
max_sharpe = mv.maximum_sharpe(constraints)

print('Markowitz Portfolios (long-only, max 20% per asset):')
print(f"  Min Variance:  Return={min_var['return']:.2%}, Vol={min_var['volatility']:.2%}, Sharpe={min_var['sharpe']:.2f}")
print(f"  Max Sharpe:    Return={max_sharpe['return']:.2%}, Vol={max_sharpe['volatility']:.2%}, Sharpe={max_sharpe['sharpe']:.2f}")

In [ ]:
# Efficient Frontier
frontier = mv.efficient_frontier(n_points=50, constraints=constraints)

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(frontier['Volatility'] * 100, frontier['Return'] * 100, 'b-', lw=2, label='Efficient Frontier')

# Plot individual assets
for ticker in investable:
    vol = np.sqrt(cov_annual.loc[ticker, ticker]) * 100
    ret = expected_returns[ticker] * 100
    ac = class_map.get(ticker, 'unknown')
    colors = {'us_equity_sectors': '#2196F3', 'international_equity': '#4CAF50',
              'crypto': '#FF9800', 'commodities': '#9C27B0',
              'fixed_income': '#607D8B', 'reits': '#E91E63'}
    ax.scatter(vol, ret, c=colors.get(ac, '#333'), s=40, alpha=0.6, zorder=5)
    ax.annotate(ticker, (vol, ret), fontsize=6, alpha=0.7)

# Plot key portfolios
for port, marker, color in [(ew, 's', 'black'), (min_var, 'D', 'green'), (max_sharpe, '*', 'red')]:
    ax.scatter(port['volatility']*100, port['return']*100, marker=marker,
               c=color, s=200, zorder=10, edgecolors='white', linewidth=2,
               label=port['name'])

ax.set_xlabel('Annualized Volatility (%)', fontsize=13)
ax.set_ylabel('Annualized Return (%)', fontsize=13)
ax.set_title('Efficient Frontier with Optimal Portfolios', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. Hierarchical Risk Parity (HRP)

In [ ]:
from project.optimization import HRPOptimizer

hrp = HRPOptimizer(returns, cov_matrix=cov_annual)
hrp_result = hrp.optimize(method='single')

print(f"HRP: Return={hrp_result['return']:.2%}, Vol={hrp_result['volatility']:.2%}, Sharpe={hrp_result['sharpe']:.2f}")
print(f"Active assets: {hrp_result['n_assets']}, Max weight: {hrp_result['max_weight']:.2%}")

## 5. Risk Parity (Equal Risk Contribution)

In [ ]:
from project.optimization import RiskParityOptimizer

rp = RiskParityOptimizer(cov_annual, expected_returns=expected_returns)
rp_result = rp.optimize()
iv_result = rp.inverse_volatility()

print(f"Risk Parity: Return={rp_result['return']:.2%}, Vol={rp_result['volatility']:.2%}, Sharpe={rp_result['sharpe']:.2f}")
print(f"Inv Vol:     Return={iv_result['return']:.2%}, Vol={iv_result['volatility']:.2%}, Sharpe={iv_result['sharpe']:.2f}")

# Risk contribution check
rc = rp_result['risk_contributions']
print(f'\nRisk Contribution Range: [{rc.min():.4f}, {rc.max():.4f}] (target: {1/len(investable):.4f})')

## 6. Mean-CVaR Optimization

In [ ]:
from project.optimization import CVaROptimizer

cvar_opt = CVaROptimizer(returns, expected_returns=expected_returns, alpha=0.05)
min_cvar = cvar_opt.minimum_cvar(constraints)

print(f"Min CVaR: Return={min_cvar['return']:.2%}, Vol={min_cvar['volatility']:.2%}, "
      f"Sharpe={min_cvar['sharpe']:.2f}")
print(f"Daily CVaR(5%): {min_cvar['cvar_daily']:.4f}, VaR(5%): {min_cvar['var_daily']:.4f}")

In [ ]:
# Mean-CVaR Efficient Frontier
cvar_frontier = cvar_opt.mean_cvar_efficient_frontier(n_points=30, constraints=constraints)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(cvar_frontier['CVaR_Daily'] * 100, cvar_frontier['Return'] * 100,
        'r-o', lw=2, markersize=4, label='Mean-CVaR Frontier')
ax.set_xlabel('Daily CVaR (5%) — %', fontsize=13)
ax.set_ylabel('Annualized Return (%)', fontsize=13)
ax.set_title('Mean-CVaR Efficient Frontier', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Strategy Comparison Dashboard

In [ ]:
# Collect all results
all_portfolios = {
    'Equal Weight': ew,
    'Min Variance': min_var,
    'Max Sharpe': max_sharpe,
    'HRP': hrp_result,
    'Risk Parity': rp_result,
    'Min CVaR': min_cvar,
    'Inv Volatility': iv_result,
}

summary = pd.DataFrame({
    name: {
        'Return (%)': p['return'] * 100,
        'Volatility (%)': p['volatility'] * 100,
        'Sharpe': p['sharpe'],
        'Active Assets': p['n_assets'],
        'Max Weight (%)': p['max_weight'] * 100,
        'HHI': p['concentration'],
    }
    for name, p in all_portfolios.items()
}).T

print('Portfolio Strategy Comparison (IN-SAMPLE — see Module 7 for out-of-sample):')
print('=' * 90)
print(summary.round(2).to_string())

In [ ]:
# Risk-Return scatter of strategies
fig, ax = plt.subplots(figsize=(12, 8))

colors_strat = plt.cm.Set1(np.linspace(0, 1, len(all_portfolios)))
for idx, (name, p) in enumerate(all_portfolios.items()):
    ax.scatter(p['volatility']*100, p['return']*100, s=200,
               c=[colors_strat[idx]], edgecolors='white', linewidth=2,
               zorder=10, label=name)

# Efficient frontier
ax.plot(frontier['Volatility']*100, frontier['Return']*100, 'k--', alpha=0.3, lw=1)

ax.set_xlabel('Annualized Volatility (%)', fontsize=13)
ax.set_ylabel('Annualized Return (%)', fontsize=13)
ax.set_title('Portfolio Strategies: Risk-Return Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Weight comparison heatmap
weight_df = pd.DataFrame({name: p['weights'] for name, p in all_portfolios.items()})

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(weight_df * 100, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Weight (%)'})
ax.set_title('Portfolio Weights by Strategy (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Strategy')
plt.tight_layout()
plt.show()

In [ ]:
# Asset class allocation per strategy
ac_alloc = {}
for name, p in all_portfolios.items():
    w = p['weights']
    ac_weights = {}
    for ac in set(class_map.values()):
        if ac == 'signals':
            continue
        tickers_in_class = [t for t, c in class_map.items() if c == ac and t in w.index]
        ac_weights[ac] = w[tickers_in_class].sum()
    ac_alloc[name] = ac_weights

ac_df = pd.DataFrame(ac_alloc).T * 100

fig, ax = plt.subplots(figsize=(14, 6))
ac_df.plot(kind='bar', stacked=True, ax=ax, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Allocation (%)')
ax.set_title('Asset Class Allocation by Strategy', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8. Export Optimized Portfolios

In [ ]:
# Export all portfolio weights
weight_df.to_parquet(f'{data_dir}/portfolio_weights.parquet')
print(f'Portfolio weights saved to {data_dir}/portfolio_weights.parquet')

# Export summary
summary.to_parquet(f'{data_dir}/portfolio_summary.parquet')
print(f'Portfolio summary saved to {data_dir}/portfolio_summary.parquet')

---

## Key Takeaways from Module 4

1. **Markowitz Max Sharpe** is useful as a diagnostic but is highly sensitive to expected-return error.
2. **HRP** provides diversified weights without matrix inversion and is more robust to estimation error.
3. **Risk Parity** ensures no single asset dominates portfolio risk.
4. **Inverse Volatility** is a transparent benchmark for volatility-scaled allocation.
5. **Mean-CVaR** focuses on tail risk and is relevant when downside loss matters.
6. **Equal weight** remains a tough benchmark and must be compared with walk-forward evidence.

### Next: Module 5 ? Risk Analysis

VaR/CVaR computation, drawdown analysis, tail risk metrics, and factor decomposition.
